# MovieLens small: exploratory data analysis

This notebook inspects the MovieLens small ratings and movie catalog listed in `docs/04_DATA.md`. It reports coverage and data quality, plots the rating distribution and most-rated movies, and does not fit or evaluate a recommendation model. The recommender evaluation uses a later per-user temporal holdout so test ratings are not used for fitting.

Download the dataset with `python src/download_data.py` from the project root before running the cells.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.eda import load_movielens, genre_popularity, summarize_ratings, top_movies
ratings, movies = load_movielens(PROJECT_ROOT / 'data' / 'ml-latest-small')
summary = summarize_ratings(ratings, movie_count=len(movies))
pd.Series(summary, name='value').to_frame()

## Rating distribution

Ratings use MovieLens' half-star scale from 0.5 to 5.0.

In [ ]:
rating_counts = ratings['rating'].value_counts().sort_index()
ax = rating_counts.plot.bar(color='#6958d6', figsize=(8, 4), rot=0)
ax.set(title='MovieLens rating distribution', xlabel='Rating', ylabel='Number of ratings')
plt.tight_layout()
plt.show()

## Most-rated titles

Rating count measures exposure, not recommendation quality or preference strength.

In [ ]:
popular = top_movies(ratings, movies, n=10)
display(popular)
popular.sort_values('rating_count').plot.barh(
    x='title', y='rating_count', legend=False, color='#e87954', figsize=(9, 5)
)
plt.title('Most-rated movies')
plt.xlabel('Number of ratings')
plt.ylabel('')
plt.tight_layout()
plt.show()

## Catalog genres

Movies can belong to multiple genres, so a title contributes once to each listed genre.

In [ ]:
genres = genre_popularity(movies)
display(genres.head(10))
genres.head(10).sort_values('movie_count').plot.barh(
    x='genre', y='movie_count', legend=False, color='#4d9a91', figsize=(9, 5)
)
plt.title('Most common movie genres')
plt.xlabel('Number of catalog titles')
plt.ylabel('')
plt.tight_layout()
plt.show()

## Data quality notes

The summary checks missing values, duplicate `(userId, movieId)` pairs, rating bounds, and catalog sparsity. These measurements describe the input only. Model selection and all reported recommender metrics are computed later from a per-user chronological train/test split.